# Boletín de tráfico DGT — explicado línea a línea

Este notebook reproduce y comenta la versión actual de `boletin_estado_carreteras.py` (con reintentos ante saturación de Gemini y detalle ampliado para Zaragoza: sentido, carril y punto kilométrico), y separa la **generación** del boletín del **envío a Telegram**, para poder disparar el envío manualmente desde aquí en lugar de depender solo de la GitHub Action.

Estructura del notebook:
1. Instalación de dependencias
2. Credenciales (se piden de forma segura, no se guardan en el archivo)
3. Funciones de fecha/hora y saludo
4. Extracción de sentido, carril y punto kilométrico
5. Extracción y limpieza de incidencias DGT (con detalle extra para Zaragoza)
6. Generación de audio (TTS)
7. Llamada a Gemini con reintentos ante error 503
8. Envío a Telegram
9. Función que genera el boletín completo (texto + audio) sin enviarlo
10. Celda para **generar** el boletín
11. Celda para **escuchar/leer** el resultado antes de enviarlo
12. Celda para **enviar manualmente** a Telegram

## 1. Instalación de dependencias

`lxml` para parsear el XML de la DGT, `google-genai` para llamar a Gemini, `pydub` para acelerar el audio (necesita `ffmpeg` instalado en el sistema, por eso el `apt-get`).

In [ ]:
!apt-get -qq install -y ffmpeg
!pip install -q requests lxml pydub google-genai

## 2. Imports

- `re`: expresiones regulares para limpiar textos.
- `time`: para las esperas entre reintentos a Gemini.
- `datetime` + `zoneinfo`: fecha/hora, usando la zona horaria real de Madrid (se ajusta sola a horario de invierno/verano).
- `requests`: llamadas HTTP (a la DGT, a Google Translate TTS y a Telegram).
- `defaultdict`: para agrupar incidencias por región sin comprobar antes si la clave existe.
- `etree` de `lxml`: parsear el XML DATEX II de la DGT.
- `genai` y `genai_errors`: cliente de la API de Gemini y sus excepciones (para detectar el error 503).
- `AudioSegment` de `pydub`: manipular el audio generado (acelerarlo).

In [ ]:
import re
import time
import datetime
from zoneinfo import ZoneInfo
import requests
from collections import defaultdict
from lxml import etree
from google import genai
from google.genai import errors as genai_errors
from pydub import AudioSegment
from IPython.display import Audio, display

## 3. Credenciales

En el script original se leen de variables de entorno (`os.environ.get(...)`), como las inyecta la GitHub Action mediante *secrets*. Aquí, para ejecutarlo a mano en Colab, las pedimos con `getpass` (no se muestran en pantalla ni quedan escritas en el notebook).

Si prefieres no teclearlas cada vez, guárdalas como *Secrets* de Colab (icono de llave 🔑 en el panel izquierdo) y sustituye esta celda por `from google.colab import userdata; GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')`, etc.

In [ ]:
from getpass import getpass

GEMINI_API_KEY = getpass("GEMINI_API_KEY: ")
TELEGRAM_BOT_TOKEN = getpass("TELEGRAM_BOT_TOKEN: ")
TELEGRAM_CHAT_ID = getpass("TELEGRAM_CHAT_ID: ")

## 4. Fecha, hora y saludo

- `DIAS_ES` / `MESES_ES`: traducen día de la semana y mes a español (Python no lo hace de forma nativa sin configurar el *locale* del sistema).
- `obtener_hora_madrid()`: hora actual ya localizada en `Europe/Madrid` (con `ZoneInfo`, corrige automáticamente el cambio de hora de invierno/verano — a diferencia de sumar horas fijas a mano, que falla medio año).
- `obtener_fecha_hora_generacion()`: construye la frase tipo *"sábado 19 de septiembre, 10:15 horas"* que se antepone al boletín.
- `obtener_saludo_y_momento()`: según la hora de Madrid, decide "Buenos días", "Buenas tardes" o "Buenas noches".

In [ ]:
DIAS_ES = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]
MESES_ES = [
    "enero", "febrero", "marzo", "abril", "mayo", "junio",
    "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"
]


def obtener_hora_madrid():
    return datetime.datetime.now(ZoneInfo("Europe/Madrid"))


def obtener_fecha_hora_generacion():
    ahora = obtener_hora_madrid()
    dia_semana = DIAS_ES[ahora.weekday()]
    mes = MESES_ES[ahora.month - 1]
    return f"{dia_semana} {ahora.day} de {mes}, {ahora.hour:02d}:{ahora.minute:02d} horas"


def obtener_saludo_y_momento():
    hora = obtener_hora_madrid().hour
    if 6 <= hora < 12:
        return "Buenos días"
    elif 12 <= hora < 20:
        return "Buenas tardes"
    else:
        return "Buenas noches"

## 5. Sentido, carril y punto kilométrico

Consultando la especificación DATEX II de la DGT, estos datos vienen en etiquetas concretas del XML: `tpegDirection` (rumbo cardinal, p.ej. `southEastBound`), `tpegDirectionRoad` (sentido de la kilometración: `positive`/`negative`/`both`), `laneUsage` (carril afectado) y `kilometerPoint`.

`extraer_sentido_carril_pk(record)` va a buscar esas etiquetas concretas con xpath (`local-name()` porque el XML usa varios namespaces) en vez de intentar adivinarlas del texto suelto del registro, que es frágil y (como pasaba antes) acaba metiendo esos valores como si fueran nombres de municipio.

- Traduce los rumbos cardinales y los valores de sentido/carril a español.
- Si hay más de un punto kilométrico (inicio y fin de un tramo), devuelve el rango; si solo hay uno, ese punto.
- Devuelve `(sentido, carriles, pk_str)`, cada uno `None`/lista vacía si no hay dato.

In [ ]:
def extraer_sentido_carril_pk(record):
    direcciones_cardinales = {
        "NORTHBOUND": "sentido norte", "SOUTHBOUND": "sentido sur",
        "EASTBOUND": "sentido este", "WESTBOUND": "sentido oeste",
        "NORTHEASTBOUND": "sentido noreste", "NORTHWESTBOUND": "sentido noroeste",
        "SOUTHEASTBOUND": "sentido sureste", "SOUTHWESTBOUND": "sentido suroeste",
    }
    sentidos_kilometracion = {
        "POSITIVE": "sentido creciente", "NEGATIVE": "sentido decreciente", "BOTH": "ambos sentidos",
    }
    carriles_traducidos = {
        "RIGHTLANE": "carril derecho", "LEFTLANE": "carril izquierdo", "CENTRALLANE": "carril central",
        "HARDSHOULDER": "arcén", "SHOULDERLANE": "arcén", "BUSLANE": "carril bus",
        "ALLLANESCOMPLETECARRIAGEWAY": "todos los carriles",
    }

    partes_sentido = []
    for tag in ("tpegDirection", "tpegDirectionRoad"):
        for v in record.xpath(f'.//*[local-name()="{tag}"]/text()'):
            v_upper = v.strip().upper()
            if v_upper in direcciones_cardinales and direcciones_cardinales[v_upper] not in partes_sentido:
                partes_sentido.append(direcciones_cardinales[v_upper])
            elif v_upper in sentidos_kilometracion and sentidos_kilometracion[v_upper] not in partes_sentido:
                partes_sentido.append(sentidos_kilometracion[v_upper])
    sentido = " / ".join(partes_sentido) if partes_sentido else None

    carriles = []
    for v in record.xpath('.//*[local-name()="laneUsage"]/text()'):
        v_upper = v.strip().upper()
        if v_upper in carriles_traducidos and carriles_traducidos[v_upper] not in carriles:
            carriles.append(carriles_traducidos[v_upper])

    puntos_km = sorted(set(record.xpath('.//*[local-name()="kilometerPoint"]/text()')))
    if not puntos_km:
        pk_str = None
    elif len(puntos_km) == 1:
        pk_str = f"pk {puntos_km[0]}"
    else:
        pk_str = f"entre pk {puntos_km[0]} y pk {puntos_km[-1]}"

    return sentido, carriles, pk_str

## 6. Extracción y limpieza de incidencias

### `limpiar_y_extraer_detalles(record)`
Recibe un `situationRecord` (un bloque XML con una incidencia) y saca de él algo legible:

- `traducciones_causa`: traduce los códigos DATEX II a español, incluyendo incidencias leves como `TRAFFICCONGESTION` → "Retención" y `OBSTRUCTION` → "Obstáculo en la vía".
- `valores_estructurales`: conjunto de valores (sentido, carril, `unknown`...) que antes se colaban como si fueran nombres de municipio; ahora se descartan aquí explícitamente, y también cualquier texto que termine en `bound` (rumbos cardinales).
- El bucle recorre todo el texto del registro y separa: causas conocidas, provincia (si es Teruel/Zaragoza/Huesca/Navarra/La Rioja) y municipios (todo lo demás que parezca un nombre de lugar).
- Llama a `extraer_sentido_carril_pk` para obtener el detalle estructurado.
- **Si la provincia es "Zaragoza"** y hay detalle disponible, lo añade al resumen: punto kilométrico, sentido y carril(es) afectado(s). El resto de provincias se queda con el resumen básico de siempre (causa + ubicación), para no sobrecargar esas zonas.

In [ ]:
def limpiar_y_extraer_detalles(record):
    traducciones_causa = {
        "ROADWORKS": "Obras", "ROADMAINTENANCE": "Mantenimiento",
        "CARRIAGEWAYCLOSURE": "Corte total de calzada", "ACCIDENT": "Accidente",
        "POORWEATHERCONDITIONS": "Meteorología adversa", "SNOW": "Nieve",
        "ICE": "Hielo", "FLOODING": "Inundación", "OBSTRUCTION": "Obstáculo en la vía",
        "TRAFFICCONGESTION": "Retención"
    }
    valores_estructurales = {
        'unspecifiedcarriageway', 'unknown', 'both', 'negative', 'positive',
        'rightlane', 'leftlane', 'centrallane', 'hardshoulder', 'shoulderlane',
        'buslane', 'alllanescompletecarriageway'
    }

    raw_texts = [t.strip() for t in record.xpath('.//text()') if t.strip()]
    municipios, provincia, causas = [], "", []

    for t in raw_texts:
        t_clean = t.strip()
        if re.match(r'^\d{4}-\d{2}-\d{2}', t_clean) or re.match(r'^-?\d+\.\d+$', t_clean) or re.match(r'^[A-Z0-9_]{8,}$', t_clean):
            continue
        t_lower = t_clean.lower()
        if t_lower in ['true', 'false', 'dgt', 'certain', 'active', 'segment', 'mandatory', 'anyvehicle']:
            continue
        if t_lower in valores_estructurales or t_lower.endswith('bound'):
            continue

        t_upper = t_clean.upper()
        if t_upper in traducciones_causa:
            causas.append(traducciones_causa[t_upper])
            continue

        if t_clean in ["Teruel", "Zaragoza", "Huesca", "Navarra", "La Rioja"]:
            provincia = t_clean
        elif len(t_clean) > 2 and not t_clean.replace('.', '').isdigit() and t_clean not in ["Aragón", "Comunidad Foral de Navarra"]:
            if t_clean not in municipios:
                municipios.append(t_clean)

    sentido, carriles, pk_str = extraer_sentido_carril_pk(record)

    ubicacion_str = f"Entre/En: {', '.join(municipios)}" if municipios else "Tramo local"
    causa_str = f"Incidencia: {', '.join(set(causas))}" if causas else "Afección en la vía"

    detalle_extra = []
    if pk_str:
        detalle_extra.append(pk_str.capitalize())
    if sentido:
        detalle_extra.append(f"Sentido: {sentido}")
    if carriles:
        detalle_extra.append(f"Carril(es): {', '.join(carriles)}")

    if provincia == "Zaragoza" and detalle_extra:
        return f"{provincia} | {ubicacion_str} | {causa_str} | " + " | ".join(detalle_extra)
    return f"{provincia} | {ubicacion_str} | {causa_str}"


def obtener_incidencias_texto():
    url = "https://nap.dgt.es/datex2/v3/dgt/SituationPublication/datex2_v37.xml"
    headers = {'User-Agent': 'Mozilla/5.0'}

    regiones_mapa = {
        "ARAGÓN": ["ZARAGOZA", "HUESCA", "TERUEL", "ARAGON", "ARAGÓN"],
        "COMUNIDAD FORAL DE NAVARRA": ["NAVARRA", "PAMPLONA"],
        "LA RIOJA": ["RIOJA", "LOGROÑO"]
    }

    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()

        parser = etree.XMLParser(recover=True, encoding='utf-8')
        root = etree.fromstring(response.content, parser=parser)

        incidencias_por_zona = defaultdict(list)

        for record in root.xpath('//*[local-name()="situationRecord"]'):
            roads = record.xpath('.//*[local-name()="roadName"]/text()')
            road_name = roads[0].strip() if roads else "Vía local"

            raw_texts = [t.strip() for t in record.xpath('.//text()') if t.strip()]
            texto_evaluacion = f"{road_name} " + " ".join(raw_texts).upper()

            region_encontrada = None
            for region, terminos in regiones_mapa.items():
                if any(term in texto_evaluacion for term in terminos):
                    region_encontrada = region
                    break

            if region_encontrada:
                resumen = limpiar_y_extraer_detalles(record)
                incidencias_por_zona[region_encontrada].append(f"- Código oficial: {road_name} -> {resumen}")

        texto_resultado = ""
        for reg in ["ARAGÓN", "COMUNIDAD FORAL DE NAVARRA", "LA RIOJA"]:
            if reg in incidencias_por_zona:
                texto_resultado += f"\n--- REGIÓN: {reg} ---\n" + "\n".join(incidencias_por_zona[reg]) + "\n"

        return texto_resultado if texto_resultado else "Sin incidencias."

    except Exception as e:
        return f"Error extrayendo datos: {e}"

## 7. Generación de audio (texto a voz)

- `acelerar_audio(...)`: carga el mp3 con `pydub`, sube el `frame_rate` (velocidad) multiplicándolo por `velocidad` y luego lo "reetiqueta" al frame rate original — así suena más rápido (acelerado simple, no *time-stretch* puro, pero suficiente para un boletín hablado).
- `generar_voz_espanol(...)`:
  - Limpia asteriscos/almohadillas/guiones bajos que pudieran colarse del texto generado por la IA.
  - Trocea el texto en frases y, si una es muy larga (>180 caracteres), la subdivide más — el endpoint de Google Translate TTS que usa tiene un límite de longitud por petición.
  - Pide el audio fragmento a fragmento y concatena los bytes en `bytes_audio_totales`.
  - Guarda ese audio "crudo" y llama a `acelerar_audio` para producir el archivo final.

In [ ]:
def acelerar_audio(archivo_entrada, archivo_salida, velocidad=1.25):
    audio = AudioSegment.from_file(archivo_entrada)
    audio_rapido = audio._spawn(audio.raw_data, overrides={"frame_rate": int(audio.frame_rate * velocidad)})
    audio_rapido = audio_rapido.set_frame_rate(audio.frame_rate)
    audio_rapido.export(archivo_salida, format="mp3")


def generar_voz_espanol(texto, archivo_salida="boletin_trafico.mp3", velocidad=1.25):
    texto_limpio = re.sub(r'[*#\_]', '', texto)
    partes = re.split(r'(?<=[.?!])\s+', texto_limpio)

    fragmentos = []
    for parte in partes:
        if len(parte) > 180:
            fragmentos.extend(re.split(r'(?<=[,;])\s+', parte))
        else:
            fragmentos.append(parte)

    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    bytes_audio_totales = bytearray()

    for fragmento in fragmentos:
        fragmento = fragmento.strip()
        if not fragmento:
            continue
        base_url = "https://translate.google.com/translate_tts"
        params = {"ie": "UTF-8", "q": fragmento, "tl": "es", "client": "tw-ob"}
        res = requests.get(base_url, params=params, headers=headers)
        if res.status_code == 200:
            bytes_audio_totales.extend(res.content)

    if bytes_audio_totales:
        temp_file = "temp_boletin.mp3"
        with open(temp_file, "wb") as f:
            f.write(bytes_audio_totales)
        acelerar_audio(temp_file, archivo_salida, velocidad=velocidad)
        return True
    return False

## 8. Llamada a Gemini con reintentos

`generar_contenido_con_reintentos(...)` envuelve la llamada a Gemini para tolerar el error `503 UNAVAILABLE` ("modelo con alta demanda"), que es intermitente por parte de Google. Reintenta hasta 4 veces con espera creciente (15 s, 30 s, 45 s) antes de propagar el error hacia arriba.

In [ ]:
def generar_contenido_con_reintentos(client, model, contents, intentos=4, espera_inicial=15):
    ultimo_error = None
    for intento in range(1, intentos + 1):
        try:
            return client.models.generate_content(model=model, contents=contents)
        except genai_errors.ServerError as e:
            ultimo_error = e
            print(f"Aviso: Gemini no disponible (intento {intento}/{intentos}): {e}")
            if intento < intentos:
                time.sleep(espera_inicial * intento)
    raise ultimo_error

## 9. Envío a Telegram

`enviar_a_telegram(archivo_audio, texto_transcripcion)`:
- Construye la URL del método `sendAudio` de la API de Bot de Telegram.
- Usa el texto del boletín como `caption`, truncado a 1024 caracteres si hace falta (límite de Telegram).
- Abre el mp3 en binario y hace un `POST` con el audio adjunto.

Esta es la función que en la GitHub Action corre sola. Aquí la aislamos para poder llamarla **a mano**, más abajo.

In [ ]:
def enviar_a_telegram(archivo_audio, texto_transcripcion):
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendAudio"
    caption = f"🎙️ **Boletín de Tráfico DGT**\n\n{texto_transcripcion}"
    if len(caption) > 1024:
        caption = caption[:1020] + "..."

    with open(archivo_audio, "rb") as audio:
        payload = {"chat_id": TELEGRAM_CHAT_ID, "caption": caption, "parse_mode": "Markdown"}
        files = {"audio": audio}
        r = requests.post(url, data=payload, files=files)

    if r.status_code == 200:
        print("✅ Enviado a Telegram correctamente.")
    else:
        print(f"⚠️ Telegram devolvió un error ({r.status_code}): {r.text}")
    return r

## 10. Generar el boletín (sin enviarlo todavía)

Equivalente al `main()` del script, pero se detiene justo antes del envío a Telegram. Construye el prompt para Gemini con el detalle ampliado de Zaragoza (sentido, carril, PK) y el resto de zonas más resumido, antepone la fecha/hora de generación y genera el audio.

Devuelve `(archivo_mp3, texto_informe)` para poder revisarlos antes de decidir enviarlos.

In [ ]:
def generar_boletin_completo():
    datos_trafico = obtener_incidencias_texto()
    if "Sin incidencias" in datos_trafico or "Error" in datos_trafico:
        print(f"No hay boletín que generar: {datos_trafico}")
        return None, None

    saludo_dinamico = obtener_saludo_y_momento()
    fecha_hora_str = obtener_fecha_hora_generacion()

    client = genai.Client(api_key=GEMINI_API_KEY)

    prompt = f"""
Eres un locutor de radio experto en información de tráfico y tiempo regional. Genera un boletín locutado fluido (200-260 palabras) para ser leído en voz alta sobre las incidencias en tiempo real de la DGT para Aragón, Navarra y La Rioja.

REGLAS DE ESTRUCTURA Y FORMATO:
1. Saludo dinámico: Comienza obligatoriamente con el saludo "{saludo_dinamico}". No menciones ni inventes tú la fecha o la hora: ya se añaden aparte, antes de tu texto.
2. Apunte meteorológico rápido: Tras el saludo, incluye una frase muy breve (10-15 palabras) sobre la situación del tiempo en el valle del Ebro y la zona norte.
3. Zaragoza (provincia), con mucho más detalle: dedica la parte más extensa y detallada del boletín a las incidencias cuyo dato de provincia sea "Zaragoza". Incluye TODAS las incidencias de Zaragoza que aparezcan en los datos, también las leves (retenciones, objetos en la vía, obras, mantenimiento), no solo accidentes o cortes graves. Cuando el dato venga acompañado de sentido de circulación, carril afectado o punto kilométrico (verás campos como "Sentido:", "Carril(es):" o "Pk"), menciónalos explícitamente (ej. "en sentido Zaragoza, carril derecho, a la altura del kilómetro 12"), porque es información práctica para quien conduce por trabajo en la zona.
4. Resto de zonas (Huesca, Teruel, Navarra, La Rioja): resume de forma más breve y ágil, mencionando incidencias graves y también las leves si son relevantes, pero sin entrar en el mismo nivel de detalle que en Zaragoza, para no alargar en exceso esas partes.
5. Sin marcas visuales: NO uses emojis, asteriscos (*) ni encabezados markdown (##).
6. Nombres conocidos: Asocia nombres populares a las carreteras (ej. "Autovía de Logroño", "Carretera de Belate", "Ronda de Zaragoza").

DATOS DGT:
{datos_trafico}
"""

    try:
        response = generar_contenido_con_reintentos(client, model='gemini-3.6-flash', contents=prompt)
    except genai_errors.ServerError as e:
        print(f"Gemini siguió sin estar disponible tras varios reintentos: {e}")
        return None, None

    texto_generado = response.text.strip()
    texto_informe = f"Boletín de tráfico actualizado el {fecha_hora_str}. {texto_generado}"

    archivo_mp3 = "boletin_trafico.mp3"
    if generar_voz_espanol(texto_informe, archivo_mp3, velocidad=1.25):
        print("✅ Boletín generado. Revísalo abajo antes de enviarlo.")
        return archivo_mp3, texto_informe

    print("⚠️ No se pudo generar el audio.")
    return None, texto_informe

## 11. Ejecutar la generación

Ejecuta esta celda cada vez que quieras un boletín nuevo. Guarda el resultado en `archivo_mp3_generado` y `texto_informe_generado`, que usará la celda de envío más abajo.

In [ ]:
archivo_mp3_generado, texto_informe_generado = generar_boletin_completo()
print(texto_informe_generado)

### (Opcional) Escuchar el audio generado antes de enviarlo

In [ ]:
if archivo_mp3_generado:
    display(Audio(archivo_mp3_generado))
else:
    print("No hay audio generado todavía. Ejecuta la celda de generación primero.")

## 12. Envío manual a Telegram

Esta es la parte que en el repo ahora corre sola dentro de la GitHub Action. Aquí la ejecutas tú, cuando quieras, sobre el boletín que acabas de generar y revisar arriba. No hace falta volver a generar nada: reutiliza `archivo_mp3_generado` y `texto_informe_generado`.

In [ ]:
if archivo_mp3_generado:
    enviar_a_telegram(archivo_mp3_generado, texto_informe_generado)
else:
    print("No hay nada que enviar: genera el boletín primero (celda de la sección 11).")